# OpenAQ-matched historical weather dataset

This notebook downloads **hourly ERA5 weather estimates** for the 20 OpenAQ monitoring-station coordinates and matches the date range to the original daily pollution file. It creates:

1. `openaq_station_weather_ERA5_hourly.csv` — station-hour weather records.
2. `openaq_station_weather_ERA5_daily.csv` — station-day summaries for joining to the OpenAQ daily table.
3. A ZIP bundle containing both files and a README.

**Upload the two original files when prompted:** `openaq_stations.csv` and `openaq_daily_all_stations.csv`. The downloader reads the actual minimum and maximum dates from the daily file, so the range is not hard-coded.

**Important:** OpenAQ contains daily records, not the exact hour of the daily maximum. The daily file can be matched by station and date; the hourly file is available if an event has a precise timestamp from another source.

## Step 1 — Install required libraries

Colab normally has pandas and NumPy already. This cell ensures the HTTP request library is available.

In [1]:
%pip -q install requests pandas numpy

## Step 2 — Create the downloader

The next cell writes the complete downloader into the Colab runtime. It requests ERA5 (not ERA5-Land) because ERA5 provides the wind speed, wind direction and precipitation variables needed here in one consistent hourly source.

In [2]:
%%writefile /content/build_openaq_weather_dataset.py
"""Build station-matched historical weather data for the OpenAQ Abu Dhabi dataset.

Run in Google Colab after uploading:
  1) openaq_stations.csv
  2) openaq_daily_all_stations.csv

Downloads hourly ERA5 data from Open-Meteo Historical Weather API for each
OpenAQ station coordinate over the exact min/max date range in the pollution
file. Produces hourly and daily weather CSVs plus a ZIP archive.
"""

from __future__ import annotations

import time
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests

API_URL = "https://archive-api.open-meteo.com/v1/archive"
TIMEZONE = "Asia/Dubai"
MODEL = "era5"  # ERA5 provides wind speed/direction and precipitation consistently.
HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "pressure_msl",
    "wind_speed_10m",
    "wind_direction_10m",
    "cloud_cover",
]


def locate_file(candidates: list[str], uploaded: dict[str, Any] | None = None) -> Path:
    """Find a user-uploaded file in Colab or the current working directory."""
    if uploaded:
        for name in candidates:
            if name in uploaded:
                return Path(name)
    for name in candidates:
        path = Path(name)
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find one of these files: " + ", ".join(candidates)
        + ". Upload the OpenAQ station file and daily pollution file first."
    )


def request_station_weather(session: requests.Session, station: pd.Series,
                            start_date: str, end_date: str) -> pd.DataFrame:
    params = {
        "latitude": float(station["lat"]),
        "longitude": float(station["lon"]),
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": TIMEZONE,
        "models": MODEL,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm",
    }
    last_error: Exception | None = None
    for attempt in range(1, 5):
        try:
            response = session.get(API_URL, params=params, timeout=120)
            response.raise_for_status()
            payload = response.json()
            if "hourly" not in payload or "time" not in payload["hourly"]:
                raise ValueError(f"No hourly data returned for station {station['station_id']}: {payload}")
            hourly = payload["hourly"]
            out = pd.DataFrame({"timestamp": pd.to_datetime(hourly["time"])})
            for variable in HOURLY_VARIABLES:
                out[variable] = hourly.get(variable, [np.nan] * len(out))
            out.insert(0, "station_id", int(station["station_id"]))
            out.insert(1, "station", str(station["station"]))
            out.insert(2, "latitude", float(station["lat"]))
            out.insert(3, "longitude", float(station["lon"]))
            out["grid_latitude"] = payload.get("latitude")
            out["grid_longitude"] = payload.get("longitude")
            out["grid_elevation_m"] = payload.get("elevation")
            out["weather_model"] = MODEL.upper()
            out["timezone"] = TIMEZONE
            return out
        except Exception as exc:
            last_error = exc
            if attempt < 4:
                wait = 2 ** attempt
                print(f"  Retry {attempt}/3 for station {station['station_id']} after error: {exc}. Waiting {wait}s")
                time.sleep(wait)
    raise RuntimeError(f"Failed to download weather for station {station['station_id']}") from last_error


def circular_mean_degrees(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if len(values) == 0:
        return np.nan
    radians = np.deg2rad(values)
    sin_mean = np.sin(radians).mean()
    cos_mean = np.cos(radians).mean()
    if np.isclose(sin_mean, 0) and np.isclose(cos_mean, 0):
        return np.nan
    return float(np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360)


def main() -> None:
    try:
        from google.colab import files  # type: ignore
        print("Upload the OpenAQ station CSV and the original daily pollution CSV.")
        uploaded = files.upload()
    except ImportError:
        uploaded = None

    station_path = locate_file(
        ["openaq_stations(1).csv", "openaq_stations.csv", "stations.csv"], uploaded
    )
    daily_path = locate_file(
        ["openaq_daily_all_stations(1).csv", "openaq_daily_all_stations.csv", "daily.csv"], uploaded
    )

    stations = pd.read_csv(station_path)
    pollution = pd.read_csv(daily_path, parse_dates=["day"])

    required_station_cols = {"station_id", "station", "lat", "lon"}
    missing_station_cols = required_station_cols - set(stations.columns)
    if missing_station_cols:
        raise ValueError(f"Station file is missing columns: {sorted(missing_station_cols)}")
    required_pollution_cols = {"station_id", "day", "parameter"}
    missing_pollution_cols = required_pollution_cols - set(pollution.columns)
    if missing_pollution_cols:
        raise ValueError(f"Daily pollution file is missing columns: {sorted(missing_pollution_cols)}")

    stations = stations.drop_duplicates("station_id").copy()
    stations["station_id"] = pd.to_numeric(stations["station_id"], errors="raise").astype(int)
    stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
    stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")
    stations = stations.dropna(subset=["lat", "lon"])

    start_date = pollution["day"].min().date().isoformat()
    end_date = pollution["day"].max().date().isoformat()
    all_dates = pd.date_range(start_date, end_date, freq="D")
    print(f"Pollution date range: {start_date} to {end_date} ({len(all_dates)} calendar days)")
    print(f"Stations with coordinates: {len(stations)}")
    print(f"Downloading {MODEL.upper()} hourly weather for each station...")

    session = requests.Session()
    session.headers.update({"User-Agent": "OpenAQ-AbuDhabi-Student-Research/1.0"})
    hourly_parts = []
    for i, (_, station) in enumerate(stations.iterrows(), start=1):
        print(f"[{i}/{len(stations)}] {station['station']} (ID {station['station_id']})")
        part = request_station_weather(session, station, start_date, end_date)
        hourly_parts.append(part)
        time.sleep(0.15)  # Be courteous to the public API.

    hourly_df = pd.concat(hourly_parts, ignore_index=True)
    hourly_df = hourly_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
    hourly_df["day"] = hourly_df["timestamp"].dt.date.astype(str)

    # Daily station/pollutant presence, so weather rows can be matched to actual OpenAQ records.
    pollution["station_id"] = pd.to_numeric(pollution["station_id"], errors="coerce").astype("Int64")
    pollution["day_key"] = pollution["day"].dt.strftime("%Y-%m-%d")
    presence = (
        pollution.dropna(subset=["station_id"])
        .groupby(["station_id", "day_key"])
        .agg(
            openaq_observation_present=("parameter", "size"),
            openaq_pollutants_present=("parameter", lambda x: ",".join(sorted(set(x.astype(str).str.upper())))),
        )
        .reset_index()
        .rename(columns={"day_key": "day"})
    )

    grouped = hourly_df.groupby(["station_id", "station", "latitude", "longitude", "grid_latitude", "grid_longitude", "grid_elevation_m", "day"], sort=False)
    daily_records = []
    for keys, group in grouped:
        station_id, station_name, lat, lon, grid_lat, grid_lon, grid_elev, day = keys
        row = {
            "station_id": station_id,
            "station": station_name,
            "latitude": lat,
            "longitude": lon,
            "grid_latitude": grid_lat,
            "grid_longitude": grid_lon,
            "grid_elevation_m": grid_elev,
            "day": day,
            "weather_model": MODEL.upper(),
            "timezone": TIMEZONE,
            "hourly_weather_records": int(group["temperature_2m"].notna().sum()),
            "temperature_2m_mean_C": group["temperature_2m"].mean(),
            "temperature_2m_min_C": group["temperature_2m"].min(),
            "temperature_2m_max_C": group["temperature_2m"].max(),
            "relative_humidity_2m_mean_pct": group["relative_humidity_2m"].mean(),
            "dew_point_2m_mean_C": group["dew_point_2m"].mean(),
            "precipitation_sum_mm": group["precipitation"].sum(min_count=1),
            "pressure_msl_mean_hPa": group["pressure_msl"].mean(),
            "cloud_cover_mean_pct": group["cloud_cover"].mean(),
            "wind_speed_10m_mean_kmh": group["wind_speed_10m"].mean(),
            "wind_speed_10m_max_kmh": group["wind_speed_10m"].max(),
            "wind_direction_10m_circular_mean_deg": circular_mean_degrees(group["wind_direction_10m"]),
        }
        daily_records.append(row)

    daily_df = pd.DataFrame(daily_records)
    daily_df = daily_df.merge(presence, on=["station_id", "day"], how="left")
    daily_df["openaq_observation_present"] = daily_df["openaq_observation_present"].fillna(0).astype(int)
    daily_df["openaq_pollutants_present"] = daily_df["openaq_pollutants_present"].fillna("")
    daily_df = daily_df.rename(columns={"openaq_observation_present": "openaq_pollutant_rows_present"})
    daily_df = daily_df.sort_values(["station_id", "day"]).reset_index(drop=True)

    # Keep all calendar days in the shared min/max range. Weather data is complete by design;
    # pollutant coverage is indicated separately by openaq_pollutant_rows_present.
    hourly_file = Path("openaq_station_weather_ERA5_hourly.csv")
    daily_file = Path("openaq_station_weather_ERA5_daily.csv")
    readme_file = Path("WEATHER_DATA_README.txt")
    zip_file = Path("openaq_weather_matched_to_pollution_dates.zip")

    hourly_df.to_csv(hourly_file, index=False)
    daily_df.to_csv(daily_file, index=False)

    readme = f"""OpenAQ station-matched historical weather dataset

SOURCE
Open-Meteo Historical Weather API, using ERA5 reanalysis (0.25 degree, approximately 25 km grid). ERA5 is selected rather than ERA5-Land because the required bundle includes wind speed/direction and precipitation.
API: {API_URL}
Documentation: https://open-meteo.com/en/docs/historical-weather-api
Data licence/attribution: Open-Meteo data are provided under CC BY 4.0; cite Open-Meteo and ECMWF/Copernicus ERA5 as appropriate.

MATCHING
Stations and coordinates: {station_path.name}
Pollution date source: {daily_path.name}
Date range: {start_date} through {end_date}, inclusive ({len(all_dates)} calendar days).
Stations requested: {len(stations)}
Timezone: {TIMEZONE}
Weather model: {MODEL.upper()}, global ERA5 reanalysis (~0.25 degree / ~25 km grid, hourly).

FILES
1) {hourly_file.name}: hourly weather by station and timestamp.
2) {daily_file.name}: daily summaries by station and date, suitable for merging with OpenAQ daily records on station_id + day.

VARIABLES
Hourly: temperature_2m (C), relative_humidity_2m (%), dew_point_2m (C), precipitation (mm per hour), pressure_msl (hPa), wind_speed_10m (km/h), wind_direction_10m (degrees), cloud_cover (%). Grid-cell centre coordinates and elevation are also retained.
Daily: temperature mean/min/max, mean relative humidity, mean dew point, total precipitation, mean sea-level pressure, mean cloud cover, mean/max wind speed, circular mean wind direction.

IMPORTANT LIMITATIONS
- These are ERA5 reanalysis estimates at grid-cell scale, NOT direct measurements at the OpenAQ monitoring stations.
- ERA5 is approximately 25 km resolution; nearby stations may map to the same or nearby grid cells and local street-level conditions are not represented. Requested station coordinates and actual returned grid-cell coordinates are both included.
- The OpenAQ input file contains daily records, not the exact hour when its daily maximum occurred. The daily weather file therefore matches station-days, not the precise detection hour. The hourly file is provided for follow-up analyses if an event timestamp becomes available.
- The daily output includes every station and every calendar date in the date range. The column openaq_pollutant_rows_present indicates how many OpenAQ pollutant rows exist for that station-day; openaq_pollutants_present lists those pollutant types. Zero means no OpenAQ pollutant observation for that station-day.
- Do not interpret temporal association between weather and pollution as proof of causation.
- Wind direction is summarized using a circular mean to handle the 0/360-degree boundary correctly.

Suggested join for daily OpenAQ data:
pd.merge(openaq_wide, weather_daily, on=['station_id', 'day'], how='left')
"""
    readme_file.write_text(readme, encoding="utf-8")

    with zipfile.ZipFile(zip_file, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(hourly_file)
        zf.write(daily_file)
        zf.write(readme_file)

    print("\nDONE")
    print(f"Hourly file: {hourly_file} | {len(hourly_df):,} rows")
    print(f"Daily file:  {daily_file} | {len(daily_df):,} rows")
    print(f"ZIP bundle:  {zip_file}")
    print("Daily rows with an OpenAQ station-day observation:", int((daily_df["openaq_pollutant_rows_present"] > 0).sum()))

    try:
        from google.colab import files  # type: ignore
        files.download(str(zip_file))
    except ImportError:
        print("Files saved in the current directory.")


if __name__ == "__main__":
    main()


Overwriting /content/build_openaq_weather_dataset.py


## Step 3 — Upload the OpenAQ files and build the weather tables

When prompted, upload the station-coordinate CSV and the original daily pollution CSV. The script will download the exact date span for each station, produce hourly and daily files, and download a ZIP bundle.

In [3]:
# ============================================================
# STEP 1 — DOWNLOAD WEATHER MATCHED TO OPENAQ
# Files must already be uploaded to /content in Colab.
# ============================================================

import os
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate manually uploaded files
# ------------------------------------------------------------

def find_uploaded_file(names):
    for name in names:
        path = Path("/content") / name
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Upload one of these files first: {names}"
    )

STATION_FILE = find_uploaded_file([
    "openaq_stations.csv",
    "openaq_stations (1).csv",
    "openaq_stations(1).csv"
])

POLLUTION_FILE = find_uploaded_file([
    "openaq_daily_all_stations.csv",
    "openaq_daily_all_stations (1).csv",
    "openaq_daily_all_stations(1).csv"
])

stations = pd.read_csv(STATION_FILE)
pollution = pd.read_csv(POLLUTION_FILE, parse_dates=["day"])

print("Files loaded successfully!")
print("Stations:", len(stations))
print("Pollution observations:", len(pollution))

# ------------------------------------------------------------
# 2. Define date range and API settings
# ------------------------------------------------------------

START_DATE = pollution["day"].min().strftime("%Y-%m-%d")
END_DATE = pollution["day"].max().strftime("%Y-%m-%d")

API_URL = "https://archive-api.open-meteo.com/v1/archive"

VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "pressure_msl",
    "wind_speed_10m",
    "wind_direction_10m",
    "cloud_cover"
]

stations = stations.drop_duplicates("station_id").copy()

# Each request contains four station coordinates.
BATCH_SIZE = 4
WAIT_BETWEEN_BATCHES = 20

checkpoint_dir = Path("/content/weather_batch_checkpoints_v2")
checkpoint_dir.mkdir(exist_ok=True)

print(f"\nDate range: {START_DATE} to {END_DATE}")
print(f"Stations: {len(stations)}")

# ------------------------------------------------------------
# 3. Download weather in batches with retry protection
# ------------------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "OpenAQ-Student-Research/1.0"
})

hourly_parts = []

batches = [
    stations.iloc[i:i+BATCH_SIZE].copy()
    for i in range(0, len(stations), BATCH_SIZE)
]

for batch_no, batch in enumerate(batches, start=1):

    checkpoint = checkpoint_dir / f"batch_{batch_no:02d}.csv"

    # Reuse a completed batch if available.
    if checkpoint.exists():
        cached = pd.read_csv(checkpoint)

        expected_ids = set(batch["station_id"].astype(int))
        cached_ids = set(cached["station_id"].astype(int))

        if expected_ids.issubset(cached_ids):
            print(f"[{batch_no}/{len(batches)}] Reusing saved batch")
            hourly_parts.append(cached)
            continue

    print(f"\n[{batch_no}/{len(batches)}] Downloading "
          f"{len(batch)} stations")

    params = {
        "latitude": ",".join(batch["lat"].astype(str)),
        "longitude": ",".join(batch["lon"].astype(str)),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(VARIABLES),
        "timezone": "Asia/Dubai",
        "models": "era5",
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    success = False

    for attempt in range(1, 7):

        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=300
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait = max(
                    60 * attempt,
                    int(retry_after) if retry_after and retry_after.isdigit()
                    else 0
                )

                print(f"Rate limited. Waiting {wait} seconds...")
                time.sleep(wait)
                continue

            response.raise_for_status()
            result = response.json()

            if isinstance(result, dict):
                result = [result]

            if len(result) != len(batch):
                raise ValueError("Unexpected number of returned locations")

            frames = []

            for (_, station), payload in zip(
                batch.iterrows(), result
            ):

                h = payload["hourly"]

                frame = pd.DataFrame({
                    "timestamp": pd.to_datetime(h["time"]),
                    "station_id": int(station["station_id"]),
                    "station": station["station"],
                    "latitude": station["lat"],
                    "longitude": station["lon"],
                    "grid_latitude": payload.get("latitude"),
                    "grid_longitude": payload.get("longitude"),
                })

                for variable in VARIABLES:
                    frame[variable] = h[variable]

                frames.append(frame)

            batch_data = pd.concat(frames, ignore_index=True)

            # Save immediately so successful batches are not lost.
            batch_data.to_csv(checkpoint, index=False)

            hourly_parts.append(batch_data)
            print(f"Saved {len(batch_data):,} hourly observations")

            success = True
            break

        except Exception as error:
            wait = min(120 * attempt, 600)
            print(f"Attempt {attempt}/6 failed: {error}")

            if attempt < 6:
                print(f"Waiting {wait} seconds...")
                time.sleep(wait)

    if not success:
        raise RuntimeError(
            f"Batch {batch_no} failed. "
            "Completed batches are saved; rerun to resume."
        )

    if batch_no < len(batches):
        time.sleep(WAIT_BETWEEN_BATCHES)

# ------------------------------------------------------------
# 4. Combine hourly weather records
# ------------------------------------------------------------

hourly_weather = pd.concat(hourly_parts, ignore_index=True)

hourly_weather["timestamp"] = pd.to_datetime(
    hourly_weather["timestamp"]
)

hourly_weather = (
    hourly_weather
    .drop_duplicates(["station_id", "timestamp"])
    .sort_values(["station_id", "timestamp"])
    .reset_index(drop=True)
)

hourly_weather.to_csv(
    "/content/openaq_station_weather_ERA5_hourly.csv",
    index=False
)

print("\nHourly weather:", hourly_weather.shape)

# ------------------------------------------------------------
# 5. Create daily weather summaries
# ------------------------------------------------------------

hourly_weather["day"] = (
    hourly_weather["timestamp"].dt.strftime("%Y-%m-%d")
)

def circular_mean(series):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if len(values) == 0:
        return np.nan

    radians = np.deg2rad(values)

    angle = np.arctan2(
        np.sin(radians).mean(),
        np.cos(radians).mean()
    )

    return np.rad2deg(angle) % 360

daily_weather = (
    hourly_weather
    .groupby(["station_id", "station", "day"])
    .agg(
        temperature_mean=("temperature_2m", "mean"),
        temperature_min=("temperature_2m", "min"),
        temperature_max=("temperature_2m", "max"),
        humidity_mean=("relative_humidity_2m", "mean"),
        dew_point_mean=("dew_point_2m", "mean"),
        rainfall_total=("precipitation", "sum"),
        pressure_mean=("pressure_msl", "mean"),
        wind_speed_mean=("wind_speed_10m", "mean"),
        wind_speed_max=("wind_speed_10m", "max"),
        cloud_cover_mean=("cloud_cover", "mean"),
        hourly_records=("temperature_2m", "count")
    )
    .reset_index()
)

# Daily wind direction must use circular averaging.
wind_direction = (
    hourly_weather
    .groupby(["station_id", "day"])["wind_direction_10m"]
    .apply(circular_mean)
    .reset_index(name="wind_direction_mean")
)

daily_weather = daily_weather.merge(
    wind_direction,
    on=["station_id", "day"],
    how="left"
)

daily_weather.to_csv(
    "/content/openaq_station_weather_ERA5_daily.csv",
    index=False
)

# ------------------------------------------------------------
# 6. Check output
# ------------------------------------------------------------

print("\nDOWNLOAD COMPLETE!")
print("Hourly dataset:", hourly_weather.shape)
print("Daily dataset:", daily_weather.shape)
print("Stations:", daily_weather["station_id"].nunique())

print("\nDaily weather preview:")
display(daily_weather.head())

print("\nFiles saved in /content:")
print("- openaq_station_weather_ERA5_hourly.csv")
print("- openaq_station_weather_ERA5_daily.csv")

Files loaded successfully!
Stations: 20
Pollution observations: 28714

Date range: 2022-10-29 to 2024-05-20
Stations: 20

[1/5] Downloading 4 stations
Saved 54,720 hourly observations

[2/5] Downloading 4 stations
Saved 54,720 hourly observations

[3/5] Downloading 4 stations
Saved 54,720 hourly observations

[4/5] Downloading 4 stations
Saved 54,720 hourly observations

[5/5] Downloading 4 stations
Saved 54,720 hourly observations

Hourly weather: (273600, 15)

DOWNLOAD COMPLETE!
Hourly dataset: (273600, 16)
Daily dataset: (11400, 15)
Stations: 20

Daily weather preview:


,station_id,station,day,temperature_mean,temperature_min,temperature_max,humidity_mean,dew_point_mean,rainfall_total,pressure_mean,wind_speed_mean,wind_speed_max,cloud_cover_mean,hourly_records,wind_direction_mean
0,369953,Bida Zayed-380554,2022-10-29,27.216667,22.1,33.9,64.416667,18.595833,0.0,1013.083333,10.529167,20.9,0.083333,24,46.292463
1,369953,Bida Zayed-380554,2022-10-30,26.729167,22.0,33.2,73.458333,20.504167,0.0,1013.845833,9.895833,15.5,22.041667,24,36.965472
2,369953,Bida Zayed-380554,2022-10-31,27.479167,22.2,33.7,67.250000,19.595833,0.0,1014.262500,7.879167,14.2,35.458333,24,80.167336
3,369953,Bida Zayed-380554,2022-11-01,27.820833,21.9,34.3,59.416667,17.754167,0.0,1014.350000,8.037500,13.3,14.750000,24,79.561197
4,369953,Bida Zayed-380554,2022-11-02,27.995833,22.1,34.9,49.416667,14.450000,0.0,1013.225000,8.533333,11.8,37.625000,24,90.372913



Files saved in /content:
- openaq_station_weather_ERA5_hourly.csv
- openaq_station_weather_ERA5_daily.csv


## Step 4 — Inspect the outputs

After the downloader completes, run this cell to check the two output tables.

In [4]:
import pandas as pd

hourly_weather = pd.read_csv('/content/openaq_station_weather_ERA5_hourly.csv', parse_dates=['timestamp'])
daily_weather = pd.read_csv('/content/openaq_station_weather_ERA5_daily.csv', parse_dates=['day'])

print('Hourly weather:', hourly_weather.shape)
print('Daily weather:', daily_weather.shape)
print('Date range:', daily_weather['day'].min(), 'to', daily_weather['day'].max())
print('Stations:', daily_weather['station_id'].nunique())
daily_weather.head()

Hourly weather: (273600, 15)
Daily weather: (11400, 15)
Date range: 2022-10-29 00:00:00 to 2024-05-20 00:00:00
Stations: 20


,station_id,station,day,temperature_mean,temperature_min,temperature_max,humidity_mean,dew_point_mean,rainfall_total,pressure_mean,wind_speed_mean,wind_speed_max,cloud_cover_mean,hourly_records,wind_direction_mean
0,369953,Bida Zayed-380554,2022-10-29,27.216667,22.1,33.9,64.416667,18.595833,0.0,1013.083333,10.529167,20.9,0.083333,24,46.292463
1,369953,Bida Zayed-380554,2022-10-30,26.729167,22.0,33.2,73.458333,20.504167,0.0,1013.845833,9.895833,15.5,22.041667,24,36.965472
2,369953,Bida Zayed-380554,2022-10-31,27.479167,22.2,33.7,67.250000,19.595833,0.0,1014.262500,7.879167,14.2,35.458333,24,80.167336
3,369953,Bida Zayed-380554,2022-11-01,27.820833,21.9,34.3,59.416667,17.754167,0.0,1014.350000,8.037500,13.3,14.750000,24,79.561197
4,369953,Bida Zayed-380554,2022-11-02,27.995833,22.1,34.9,49.416667,14.450000,0.0,1013.225000,8.533333,11.8,37.625000,24,90.372913
